In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
import os

def read_GSSHA_dep(folder_path, filename):
    """
    Reads a GSSHA .dep file.

    Returns
    -------
    dict
        {
            "timesteps": list,
            "depths": list of numpy arrays
        }
    """

    dep_file = os.path.join(folder_path, filename)

    timesteps = []
    depths = []

    current_depths = []
    reading = False

    with open(dep_file, "r") as f:

        for line in f:

            line = line.strip()

            if not line:
                continue

            # New timestep
            if line.startswith("TS"):

                if reading:
                    depths.append(
                        np.asarray(current_depths, dtype=float)
                    )

                current_depths = []

                timestep = float(line.split()[2])

                timesteps.append(timestep)

                reading = True

                continue

            if line == "ENDDS":
                break

            if reading:
                current_depths.append(float(line))

    # save last timestep
    if current_depths:
        depths.append(np.asarray(current_depths, dtype=float))

    return {
        "timesteps": timesteps,
        "depths": depths
    }

In [7]:
dep

{'timesteps': [0.5, 10.5, 20.5, 30.5],
 'depths': [array([0., 0., 0., ..., 0., 0., 0.]),
  array([0., 0., 0., ..., 0., 0., 0.]),
  array([0., 0., 0., ..., 0., 0., 0.]),
  array([0., 0., 0., ..., 0., 0., 0.])]}

In [2]:
cwd = os.getcwd()
script_dir = Path.cwd()

dep = read_GSSHA_dep(
    folder_path=script_dir,
    filename="Waialua_FIM_testing.dep"
)

In [4]:
import numpy as np
import pandas as pd
from collections import Counter


def calculate_dep_changes(
    dep_data,
    cell_size,
    depth_threshold=0.01
):
    """
    Create a DataFrame of depth and inundated-area changes between
    consecutive complete GSSHA DEP timesteps.

    Incomplete timestep arrays, such as the final timestep while GSSHA
    is actively writing the DEP file, are skipped.

    Parameters
    ----------
    dep_data : dict
        Output from read_GSSHA_dep():

        {
            "timesteps": [t0, t1, t2, ...],
            "depths": [
                np.ndarray,
                np.ndarray,
                ...
            ]
        }

    cell_size : float
        GSSHA grid-cell size in model distance units.
        For a 10 m grid, use cell_size=10.

    depth_threshold : float, optional
        Minimum depth used to classify a cell as inundated.
        Default is 0.01 model depth units.

    Returns
    -------
    pandas.DataFrame
        Columns:
        - timestep
        - cumulative_change
        - max_positive_change_per_cell
        - inundated_area_change
    """

    timesteps = dep_data["timesteps"]
    depths = dep_data["depths"]

    output_columns = [
        "timestep",
        "cumulative_change",
        "max_positive_change_per_cell",
        "inundated_area_change"
    ]

    if len(timesteps) != len(depths):
        raise ValueError(
            "dep_data['timesteps'] and dep_data['depths'] "
            f"must have the same length. Found {len(timesteps)} "
            f"timesteps and {len(depths)} depth arrays."
        )

    if len(depths) < 2:
        return pd.DataFrame(columns=output_columns)

    # Flatten each depth array so shape differences such as
    # (rows, columns) versus (number_of_cells,) do not matter.
    flattened_depths = [
        np.asarray(depth, dtype=np.float64).ravel()
        for depth in depths
    ]

    # Find the most common array size. This should represent the
    # fully written timestep size. A partially written final timestep
    # will normally have a smaller, uncommon size.
    array_sizes = [depth.size for depth in flattened_depths]

    expected_size = Counter(array_sizes).most_common(1)[0][0]

    complete_timesteps = []
    complete_depths = []

    for timestep, depth, array_size in zip(
        timesteps,
        flattened_depths,
        array_sizes
    ):
        if array_size != expected_size:
            print(
                f"Skipping incomplete DEP timestep {timestep}: "
                f"found {array_size:,} values; "
                f"expected {expected_size:,}."
            )
            continue

        complete_timesteps.append(timestep)
        complete_depths.append(depth)

    if len(complete_depths) < 2:
        print(
            "Fewer than two complete DEP timesteps are available. "
            "No timestep changes can be calculated yet."
        )
        return pd.DataFrame(columns=output_columns)

    cell_area = float(cell_size) ** 2
    results = []

    for i in range(1, len(complete_depths)):
        previous_depth = complete_depths[i - 1]
        current_depth = complete_depths[i]

        # Signed depth change:
        # positive = depth increased
        # negative = depth decreased
        depth_change = current_depth - previous_depth

        # Net signed depth change over the entire model.
        cumulative_change = np.nansum(
            depth_change,
            dtype=np.float64
        )

        # Largest positive increase at any grid cell.
        positive_change = np.maximum(depth_change, 0.0)

        max_positive_change_per_cell = np.nanmax(
            positive_change
        )

        # Calculate inundated cells at each complete timestep.
        previous_wet_cells = np.count_nonzero(
            previous_depth > depth_threshold
        )

        current_wet_cells = np.count_nonzero(
            current_depth > depth_threshold
        )

        inundated_area_change = (
            current_wet_cells - previous_wet_cells
        ) * cell_area

        results.append({
            "timestep": complete_timesteps[i],
            "cumulative_change": cumulative_change,
            "max_positive_change_per_cell": (
                max_positive_change_per_cell
            ),
            "inundated_area_change": inundated_area_change
        })

    return pd.DataFrame(results, columns=output_columns)

In [5]:
df_changes = calculate_dep_changes(
    dep_data=dep,
    cell_size=10,
    depth_threshold=0.01
)


Skipping incomplete DEP timestep 30.5: found 153,365 values; expected 153,438.


In [6]:
df_changes

,timestep,cumulative_change,max_positive_change_per_cell,inundated_area_change
0,10.5,282.545182,2.874514,57500.0
1,20.5,244.419298,2.623107,38200.0


In [24]:
def find_inundation_equilibrium(
    df_area,
    area_tolerance=200.0,
    consecutive_steps=5
):
    """
    Finds when inundated area remains stable for a specified
    number of consecutive output timesteps.

    Parameters
    ----------
    df_area : pandas.DataFrame
        Output from calculate_GSSHA_inundated_area().

    area_tolerance : float, optional
        Maximum allowed absolute inundated-area change between
        consecutive timesteps.

    consecutive_steps : int, optional
        Number of consecutive timesteps that must satisfy the
        tolerance.

    Returns
    -------
    dict or None
        Equilibrium information, or None if equilibrium was
        not reached.
    """

    stable_count = 0

    for i in range(1, len(df_area)):

        area_change = abs(
            df_area.loc[i, "inundated_area_change"]
        )

        if area_change <= area_tolerance:
            stable_count += 1
        else:
            stable_count = 0

        if stable_count >= consecutive_steps:
            start_index = i - consecutive_steps + 1

            return {
                "equilibrium_timestep": df_area.loc[i, "timestep"],
                "stable_period_start": df_area.loc[
                    start_index, "timestep"
                ],
                "inundated_area": df_area.loc[i, "inundated_area"],
                "inundated_cells": df_area.loc[
                    i, "inundated_cells"
                ],
                "maximum_allowed_change": area_tolerance,
                "consecutive_steps": consecutive_steps
            }

    return None

In [25]:
equilibrium = find_inundation_equilibrium(
    df_area,
    area_tolerance=400.0,
    consecutive_steps=2
)

print(equilibrium)

{'equilibrium_timestep': 600.5, 'stable_period_start': 570.5, 'inundated_area': 4897400, 'inundated_cells': 48974, 'maximum_allowed_change': 400.0, 'consecutive_steps': 2}
